# R49 — frozen-model proposal diagnosis and Soft-NMS pilot

**Choose a GPU runtime (T4 is fine) and Run all.** CPU is supported but slower. No training or model promotion occurs. This uses the exact PR137 candidate checkpoint and original 20 development photos, with R38, frozen666 (primary) and historical663 evaluated separately. The 15 known R38 misses are traced through decoded geometry, confidence, ordinary NMS, post-clipping suppression, and final assignment. If the fresh baseline differs from the saved T4 outputs, the report identifies that difference and traces the union of saved/fresh misses; the intervention comparison uses the same fresh decoded tensors, not mixed runtimes.

**Fixed hypothesis:** replace ordinary hard NMS at IoU 0.70 with linear Soft-NMS (decay score by 1−IoU only above 0.70), keeping the strict 0.13 confidence cutoff, 300 output cap, same postclip stage, 0.50 matching IoU and 0.50 endpoint threshold. The comparison is exploratory: it may worsen recall or add unmatched detections. Do not deploy a new setting on the basis of this notebook alone.

Return `baomai-r49-diagnosis-results.zip`. It contains the verified PR137 audit, fresh predictions, all decoded tensors (compressed in the ZIP), stage traces, stable row identities, paired scores, and source script. No photos or weights are in the return. Do not rerun R47 or train on held annotations for this experiment.

## 1. Fetch the pinned project source

In [ ]:
from pathlib import Path
import os, sys, subprocess, hashlib, json, zipfile
SOURCE_REVISION = "8cf5a2afc9fe28ba0f4f4596c27944761a790b76"
BASE = Path("/content/baomai-r49")
SOURCE, WORK = BASE / "source", BASE / "work"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE":"1", "GIT_TERMINAL_PROMPT":"0"}
def run(cmd,cwd=None):
    subprocess.run(list(map(str,cmd)),cwd=cwd,env=ENV,check=True)
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Nonempty source folder preserved; use a fresh BASE")
    run(["git","init",SOURCE])
    run(["git","remote","add","origin","https://github.com/Siuuuers/dl.git"],SOURCE)
    run(["git","fetch","--depth=1","--filter=blob:none","origin",SOURCE_REVISION],SOURCE)
    run(["git","sparse-checkout","init","--no-cone"],SOURCE)
    run(["git","sparse-checkout","set","/scripts/","/data/manifest.json","/data/labels/",
         "/data/candidates/front-review-2026-11-26-r37/",
         "/data/reviews/additional-fronts-2026-11-27-r47/",
         "/data/reviews/fronts-2026-11-26-r38/",
         "/data/references/sequence-refined-reference-2026-11-18/",
         "/data/references/visible-front-completion-2026-11-14/"],SOURCE)
    run(["git","checkout","--detach",SOURCE_REVISION],SOURCE)
assert subprocess.check_output(["git","rev-parse","HEAD"],cwd=SOURCE,text=True).strip()==SOURCE_REVISION
assert not subprocess.check_output(["git","status","--porcelain","--untracked-files=no"],cwd=SOURCE,text=True).strip()
print("Pinned R48 source verified:",SOURCE_REVISION)


## 2. Preserve the runtime packages and install the pinned detector

In [ ]:
from importlib.metadata import version
import torch
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("Runtime:", torch.cuda.get_device_name(0) if DEVICE == "0" else "CPU (slower, no training)")
constraints = BASE / "torch-constraints.txt"
constraints.write_text("".join(f"{name}=={version(name)}\n" for name in ("torch","torchvision")))
run([sys.executable,"-m","pip","install","-q","-c",constraints,"ultralytics==8.4.182"])
print("Ultralytics:", version("ultralytics"))


## 3. Restore the exact diagnostic script and run synthetic checks

In [ ]:
SCRIPT = BASE / "diagnose_r49.py"
SCRIPT_SHA256 = '188df92036d5dc8fd8e13fc8d8815728a56c33ced654de2cf4716788eac47811'
SCRIPT_TEXT = r'''"""R49: PR137 candidate stage tracing and one fixed linear Soft-NMS pilot."""
import argparse
from collections import Counter
import hashlib
import importlib.metadata as md
import json
from pathlib import Path
import platform
import sys
import time
import urllib.request
import zipfile

CONF, OVERLAP, MATCH, CAP = .13, .70, .50, 300
SOURCE = '8cf5a2afc9fe28ba0f4f4596c27944761a790b76'


def sha(data):
    return hashlib.sha256(data).hexdigest()


def need(ok, message):
    if not ok:
        raise RuntimeError(message)


def save(path, obj):
    path.write_text(json.dumps(obj, indent=2, allow_nan=False) + '\n')


def soft_select(boxes, confidence):
    """One fixed linear Soft-NMS: IoU>.70 -> score*(1-IoU), strict .13 gate."""
    import numpy as np
    scores = np.asarray(confidence, dtype=np.float64).copy()
    boxes = np.asarray(boxes, dtype=np.float64)
    need(boxes.shape == (len(scores), 4) and np.isfinite(boxes).all()
         and np.isfinite(scores).all(), 'Invalid decoded inputs')
    active = np.flatnonzero(scores > CONF)
    need(len(active) <= 30000, 'Native max_nms cap exceeded; pilot stopped')
    chosen, values = [], []
    area = np.prod(np.maximum(boxes[:, 2:] - boxes[:, :2], 0), axis=1)
    while len(active) and len(chosen) < CAP:
        pos = int(np.argmax(scores[active]))
        i = int(active[pos])
        chosen.append(i); values.append(float(scores[i]))
        active = np.delete(active, pos)
        intersection = np.prod(np.maximum(np.minimum(boxes[active, 2:], boxes[i, 2:]) -
                                          np.maximum(boxes[active, :2], boxes[i, :2]), 0), axis=1)
        union = area[i] + area[active] - intersection
        iou = np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)
        scores[active] *= np.where(iou > OVERLAP, 1 - iou, 1)
        active = active[scores[active] > CONF]
    return chosen, values, bool(len(active))


def stage(eligible, scores, ordinary, final, capped):
    above = {i for i in eligible if scores[i] > CONF}
    if not eligible:
        return 'no_iou_eligible_decoded_box'
    if not above:
        return 'confidence_gate'
    if not above.intersection(ordinary):
        return 'ordinary_nms_or_cap' if capped else 'ordinary_nms'
    if not above.intersection(final):
        return 'postclip_suppression'
    return 'retained_assignment'


def selftest():
    import numpy as np
    boxes = np.array([[0, 0, 10, 10], [0, 0, 10, 10], [20, 0, 30, 10], [0, 0, 10, 8]])
    chosen, scores, capped = soft_select(boxes, [.9, .8, .7, .75])
    need(chosen == [0, 2, 3] and abs(scores[2] - .15) < 1e-10 and not capped, 'Soft-NMS synthetic test failed')
    need(soft_select(boxes[:2], [CONF, .12])[0] == [], 'Strict score gate failed')
    need(stage([], [.8], set(), set(), False) == 'no_iou_eligible_decoded_box', 'Stage test')
    need(stage([0], [.1], set(), set(), False) == 'confidence_gate', 'Stage test')
    need(stage([0], [.8], set(), set(), False) == 'ordinary_nms', 'Stage test')
    need(stage([0], [.8], set(), set(), True) == 'ordinary_nms_or_cap', 'Stage test')
    need(stage([0], [.8], {0}, set(), False) == 'postclip_suppression', 'Stage test')
    need(stage([0], [.8], {0}, {0}, False) == 'retained_assignment', 'Stage test')
    print('Synthetic selection and stage checks passed')


def run(root, work, device):
    import numpy as np
    import torch
    from PIL import Image
    from ultralytics import YOLO
    from ultralytics.models.yolo.pose.predict import PosePredictor
    from ultralytics.utils import nms, ops
    from audit_additional_front_pilot_r47_results import obtain_archive, audit, REFERENCES, INPUT_SOURCE, MODEL_HASHES, ARCHIVE_SHA
    from audit_additional_front_pilot_r45_results import corner_groups
    from evaluate_pose import INFERENCE, _prediction_records, parse_pose_labels
    from pose_checkpoint import check_loaded_pose_model
    from pose_scoring import score_image, aggregate_scores, _box_iou
    from predict_pose_folder import postclip_nms

    need(md.version('ultralytics') == '8.4.182', 'Use pinned Ultralytics')
    selftest()
    work.mkdir(parents=True, exist_ok=True)
    cache = work / 'cache'; cache.mkdir(exist_ok=True)
    archive = obtain_archive(cache)
    receipt = audit(archive, root)
    need(receipt['status'] == 'passed', 'PR137 audit failed')
    with zipfile.ZipFile(archive) as z:
        prior = json.loads(z.read('evaluation/predictions.json'))['candidate']
        old_summary = json.loads(z.read('evaluation/summary.json'))
        weight = z.read('candidate/fit/weights/last.pt')
    need(sha(weight) == MODEL_HASHES['candidate'], 'Wrong PR137 checkpoint')
    weights = cache / 'pr137-candidate.pt'
    if not weights.exists(): weights.write_bytes(weight)
    need(sha(weights.read_bytes()) == MODEL_HASHES['candidate'], 'Cached checkpoint differs')
    manifest = json.loads((root / 'data/manifest.json').read_text())
    records = {r['id']: r for r in manifest['records'] if r['split'] == 'val'}
    need(len(records) == 20 and set(records) == set(prior), 'Development membership differs')
    refs = {}
    for name, (relative, expected, count) in REFERENCES.items():
        p = root / relative
        need(sha(p.read_bytes()) == expected, 'Reference manifest differs')
        refs[name] = {}
        for r in json.loads(p.read_text())['files']:
            if r.get('split', 'val') != 'val': continue
            i = r['image_id']; data = (p.parent / r['label_path']).read_bytes()
            need(sha(data) == r['label_sha256'], 'Reference label differs')
            need(r.get('image_sha256', r.get('source_image_sha256')) == records[i]['image_sha256'], 'Reference image differs')
            refs[name][i] = parse_pose_labels(data.decode(), 1280, 720, name + '/' + i)
        need(set(refs[name]) == set(records) and sum(map(len, refs[name].values())) == count, 'Reference inventory differs')
    maps_path = root / REFERENCES['r38'][0]
    maps_desc = json.loads(maps_path.read_text())['row_maps']
    maps_data = (root / maps_desc['path']).read_bytes()
    need(sha(maps_data) == maps_desc['sha256'], 'R38 row map differs')
    maps = json.loads(maps_data)

    class TracePredictor(PosePredictor):
        def postprocess(self, preds, img, orig_imgs, **kwargs):
            raw = preds[0] if isinstance(preds, (tuple, list)) else preds
            self.trace = (raw.detach().clone(), img, orig_imgs)
            return super().postprocess(preds, img, orig_imgs, **kwargs)

    torch.set_num_threads(4)
    model = YOLO(str(weights)); check_loaded_pose_model(model)
    model.predict(source=np.zeros((720, 1280, 3), dtype=np.uint8), predictor=TracePredictor, device=device, conf=CONF, **INFERENCE)
    predictor = model.predictor
    need(not predictor.model.end2end and not predictor.model.fp16, 'Expected ordinary one-to-many FP32 inference')
    def sync():
        if device != 'cpu': torch.cuda.synchronize()
    outputs, traces, cases, runtime_rows = {}, {}, [], []
    raw_tensors = work / 'decoded'; raw_tensors.mkdir(exist_ok=True)
    for i, record in sorted(records.items()):
        path = cache / (i + '.png')
        if not path.exists():
            url = f'https://media.githubusercontent.com/media/Siuuuers/dl/{INPUT_SOURCE}/data/{record["image"]}'
            with urllib.request.urlopen(url, timeout=120) as response: data = response.read()
            need(sha(data) == record['image_sha256'], 'Original download differs: ' + i)
            path.write_bytes(data)
        need(sha(path.read_bytes()) == record['image_sha256'], 'Original differs: ' + i)
        with Image.open(path) as im: need(im.size == (1280, 720), 'Original dimensions differ')
        result = model.predict(source=str(path), device=device, conf=CONF, **INFERENCE)[0]
        need(tuple(result.orig_shape) == (720, 1280), 'Prediction dimensions differ')
        primary = _prediction_records(result)
        raw, image_tensor, originals = predictor.trace
        need(tuple(raw.shape[:2]) == (1, 11), 'Unexpected decoded shape')
        raw_np = raw.cpu().numpy()
        need(np.isfinite(raw_np).all(), 'Nonfinite decoded tensor')
        np.save(raw_tensors / (i + '.npy'), raw_np, allow_pickle=False)
        input_boxes = ops.xywh2xyxy(raw[0, :4].T.clone())
        scaled = ops.scale_boxes(image_tensor.shape[2:], input_boxes.clone(), result.orig_img.shape).cpu().tolist()
        confidences = raw[0, 4].cpu().tolist()
        sync(); t = time.perf_counter()
        detections, indices = nms.non_max_suppression(raw.clone(), conf_thres=CONF, iou_thres=OVERLAP,
            classes=None, agnostic=False, max_det=CAP, nc=1, end2end=False, rotated=False, return_idxs=True)
        keep = [int(j) for j in indices[0].tolist()]
        ordinary = _prediction_records(predictor.construct_results([d.clone() for d in detections], image_tensor, originals)[0])
        need(ordinary == primary, 'Native NMS index reconstruction differs: ' + i)
        hard, hard_audit = postclip_nms(ordinary)
        sync(); hard_ms = (time.perf_counter() - t) * 1000
        hard_keep = [keep[j-1] for j in hard_audit['retained_primary_prediction_ids']]
        sync(); t = time.perf_counter()
        soft_ids, values, soft_capped = soft_select(input_boxes.cpu().numpy(), confidences)
        ix = torch.tensor(soft_ids, dtype=torch.long, device=raw.device)
        value_tensor = torch.tensor(values, dtype=raw.dtype, device=raw.device).reshape(-1, 1)
        cls = torch.zeros_like(value_tensor)
        rows = torch.cat([input_boxes[ix], value_tensor, cls, raw[0, 5:, :].T[ix]], dim=1)
        soft_raw = _prediction_records(predictor.construct_results([rows.clone()], image_tensor, originals)[0])
        soft, soft_audit = postclip_nms(soft_raw)
        sync(); soft_ms = (time.perf_counter() - t) * 1000
        soft_keep = [soft_ids[j-1] for j in soft_audit['retained_primary_prediction_ids']]
        outputs[i] = {'saved': prior[i], 'hard_raw': ordinary, 'hard': hard, 'soft_raw': soft_raw, 'soft': soft,
                      'hard_audit': hard_audit, 'soft_audit': soft_audit, 'ordinary_indices': keep,
                      'hard_indices': hard_keep, 'soft_raw_indices': soft_ids, 'soft_indices': soft_keep}
        gold = refs['r38'][i]
        score_old = score_image(gold, prior[i]['postclip'], 1280, 720, MATCH, .5)
        score_hard = score_image(gold, hard, 1280, 720, MATCH, .5)
        score_soft = score_image(gold, soft, 1280, 720, MATCH, .5)
        old_miss, hard_miss, soft_miss = [set(s['unmatched_gold_indices']) for s in (score_old, score_hard, score_soft)]
        above_count = sum(c > CONF for c in confidences)
        capped = above_count > 30000 or len(keep) == CAP
        rowmap = {r['current_row']: r for r in maps[i]['rows']}
        for j in sorted(old_miss | hard_miss):
            g = gold[j]
            overlap = [_box_iou(b, g['bbox_xyxy']) for b in scaled]
            eligible = [k for k, v in enumerate(overlap) if v >= MATCH]
            def item(k):
                return {'anchor_index': k, 'bbox_xyxy': scaled[k], 'iou': overlap[k], 'confidence': confidences[k],
                        'ordinary': k in keep, 'postclip': k in hard_keep, 'soft_postclip': k in soft_keep}
            best = max(range(len(overlap)), key=lambda k: (overlap[k], confidences[k], -k))
            cases.append({'image': i, 'r38_row': j + 1, 'stable_id': rowmap[j+1]['stable_id'], 'gold': g,
                          'saved_miss': j in old_miss, 'fresh_hard_miss': j in hard_miss, 'soft_miss': j in soft_miss,
                          'locatable_corners': sum(k[2] == 2 for k in g['keypoints']),
                          'mechanism': stage(eligible, confidences, set(keep), set(hard_keep), capped) if j in hard_miss else 'fresh_baseline_matched',
                          'best_decoded': item(best), 'eligible': [item(k) for k in eligible], 'output_cap_possible': capped})
        traces[i] = {'raw_tensor_sha256': sha(raw_np.tobytes()), 'shape': list(raw_np.shape), 'input_shape': list(image_tensor.shape),
                     'image_sha256': record['image_sha256'], 'above_confidence': above_count, 'soft_cap_reached': soft_capped,
                     'native_reconstruction_exact': True, 'saved_raw_exact': ordinary == prior[i]['raw'],
                     'saved_postclip_exact': hard == prior[i]['postclip']}
        runtime_rows.append({'image': i, 'hard_suppression_and_construct_ms': hard_ms, 'soft_suppression_and_construct_ms': soft_ms})
        print(i, 'fresh hard/soft misses:', len(hard_miss), len(soft_miss), 'unmatched:', score_hard['fp'], score_soft['fp'], flush=True)
    reports = {}
    for name, gold in refs.items():
        reports[name] = {}
        for view in ('saved_raw', 'saved', 'hard_raw', 'hard', 'soft_raw', 'soft'):
            scores, groups = {}, {str(n): Counter() for n in (0, 1, 2)}
            for i, g in gold.items():
                p = outputs[i]['saved']['raw' if view == 'saved_raw' else 'postclip'] if view.startswith('saved') else outputs[i][view]
                scores[i] = score_image(g, p, 1280, 720, MATCH, .5)
                for k, v in corner_groups(g, p, scores[i]).items(): groups[k].update(v)
            report = {'detection': aggregate_scores(list(scores.values())), 'corners_by_locatable_count': groups, 'per_image': scores}
            if view.startswith('saved'):
                need(report == old_summary['references'][name]['raw' if view == 'saved_raw' else 'postclip']['candidate'], 'Saved replay differs')
            reports[name][view] = report
        a,b = [reports[name][k]['per_image'] for k in ('hard','soft')]
        reports[name]['paired_delta'] = {i: {'recovered_rows': [j+1 for j in sorted(set(a[i]['unmatched_gold_indices'])-set(b[i]['unmatched_gold_indices']))],
                                                'newly_missed_rows': [j+1 for j in sorted(set(b[i]['unmatched_gold_indices'])-set(a[i]['unmatched_gold_indices']))]} for i in gold}
    summary = {'source': SOURCE, 'script_sha256': sha(Path(__file__).read_bytes()), 'archive_sha256': ARCHIVE_SHA,
               'checkpoint_sha256': MODEL_HASHES['candidate'], 'runtime': {'python': platform.python_version(), 'torch': torch.__version__,
               'ultralytics': md.version('ultralytics'), 'torchvision': md.version('torchvision'), 'device': device,
               'gpu': torch.cuda.get_device_name(0) if device != 'cpu' else None},
               'settings': {**INFERENCE, 'conf': CONF, 'matching_iou': MATCH, 'endpoint_conf': .5,
                            'soft_nms': 'linear, IoU > .70, score *= 1-IoU, strict conf > .13, max_det 300, same postclip'},
               'same_tensor_comparison': True, 'fresh_inference': True, 'training': False, 'automatic_promotion': False,
               'saved_baseline_exact_images': sum(t['saved_postclip_exact'] for t in traces.values()),
               'fresh_miss_mechanisms': dict(Counter(c['mechanism'] for c in cases if c['fresh_hard_miss'])),
               'reports': reports, 'timing_scope': 'Illustrative per-image suppression and result construction only; not full end-to-end latency.'}
    save(work / 'summary.json', summary); save(work / 'predictions.json', outputs); save(work / 'trace.json', traces)
    save(work / 'missed-cases.json', cases); save(work / 'timing.json', runtime_rows)
    save(work / 'input-audit.json', receipt)
    files = [work / name for name in ('summary.json','predictions.json','trace.json','missed-cases.json','timing.json','input-audit.json')] + sorted(raw_tensors.glob('*.npy'))
    need(len(files)==26, 'Return membership differs')
    script_copy = work / 'diagnose_r49.py'
    script_copy.write_bytes(Path(__file__).read_bytes())
    files.append(script_copy)
    inventory = {str(p.relative_to(work)): {'bytes': p.stat().st_size, 'sha256': sha(p.read_bytes())} for p in files}
    target = work / 'baomai-r49-diagnosis-results.zip'
    tmp = target.with_suffix('.tmp')
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in files: z.write(p, str(p.relative_to(work)))
        z.writestr('inventory.json', json.dumps(inventory, indent=2))
    with zipfile.ZipFile(tmp) as z:
        need(z.testzip() is None and set(z.namelist()) == set(inventory) | {'inventory.json'}, 'Return invalid')
        for n,v in inventory.items(): need(sha(z.read(n)) == v['sha256'], 'Return member differs')
    tmp.replace(target)
    for name, report in reports.items():
        for view in ('saved', 'hard', 'soft'):
            d=report[view]['detection']; c=report[view]['corners_by_locatable_count']['2']
            print(name, view, {k:d[k] for k in ('tp','fn','fp','duplicate_fp','other_fp')}, '5px/10px:',c['within5'],c['within10'],'/',c['eligible'])
    print('Fresh miss mechanisms:', summary['fresh_miss_mechanisms'])
    print('Return:', target, 'SHA256:', sha(target.read_bytes()))


if __name__ == '__main__':
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('--root', type=Path)
    parser.add_argument('--work', type=Path)
    parser.add_argument('--device', default='0')
    parser.add_argument('--selftest', action='store_true')
    args = parser.parse_args()
    if args.selftest: selftest()
    else: run(args.root.resolve(), args.work.resolve(), args.device)
'''
assert hashlib.sha256(SCRIPT_TEXT.encode()).hexdigest() == SCRIPT_SHA256
if SCRIPT.exists():
    assert hashlib.sha256(SCRIPT.read_bytes()).hexdigest() == SCRIPT_SHA256, "Incompatible script preserved"
else:
    SCRIPT.write_text(SCRIPT_TEXT)
run([sys.executable,"-B",SCRIPT,"--selftest"])


## 4. Run the frozen-model experiment and download the return ZIP

In [ ]:
ENV["PYTHONPATH"] = str(SOURCE / "scripts") + os.pathsep + ENV.get("PYTHONPATH", "")
ARCHIVE = WORK / "baomai-r49-diagnosis-results.zip"
if not ARCHIVE.exists():
    run([sys.executable,"-B",SCRIPT,"--root",SOURCE,"--work",WORK,"--device",DEVICE], SOURCE / "scripts")
with zipfile.ZipFile(ARCHIVE) as z:
    assert z.testzip() is None
    manifest = json.loads(z.read("inventory.json"))
    assert len(z.namelist()) == len(set(z.namelist())) and set(z.namelist()) == set(manifest) | {"inventory.json"}
    for name, entry in manifest.items():
        data=z.read(name)
        assert entry=={"bytes":len(data),"sha256":hashlib.sha256(data).hexdigest()}, name
    summary=json.loads(z.read("summary.json"))
    assert summary["source"]==SOURCE_REVISION and summary["script_sha256"]==SCRIPT_SHA256
    assert summary["archive_sha256"]=="fc8c40357c83831d17f88b0bb5a5ecd2e0cb1a8128be697e37e80620e4981023"
    print("Fresh baseline exact-image count:",summary["saved_baseline_exact_images"],"/ 20")
    print("Miss stages:",summary["fresh_miss_mechanisms"])
    for name,views in summary["reports"].items():
        for view in ("saved","hard","soft"):
            d=views[view]["detection"]; c=views[view]["corners_by_locatable_count"]["2"]
            print(name,view,"missed",d["fn"],"unmatched",d["fp"],"duplicate subset",d["duplicate_fp"],"both within 5px",c["within5"],"/",c["eligible"])
print("Return:",ARCHIVE,"SHA-256:",hashlib.sha256(ARCHIVE.read_bytes()).hexdigest())
try:
    from google.colab import files
except ImportError:
    print("Saved locally:", ARCHIVE)
else:
    files.download(str(ARCHIVE))
